# PR to PO Test Notebook

**Purpose:** Demo scope through **Section 5b** — **document upload** → **document categorization** → **document parsing** (Quote and MSA with retries and validation) → **save parsed outputs** to output/. This notebook matches the master notebook through Section 5b. No structured DB, RAG, orchestrator, or compliance checks in this notebook.

**Environment:** Same as master: set **ENV_MODE=local** or **aws** and required vars (see `.env.example`). Section 0 validates them. Run Section 0 → 1 → 2 → 3 → 4 → 5 → 5b in order.

**Getting started**

1. Copy **.env.example** to **.env** and set required vars for your mode (e.g. **GROQ_API_KEY**, **QDRANT_URL**, **QDRANT_API_KEY** for local).
2. Run **Section 0** (env detection & validation) → **Section 1** (paths + LLM) → **Section 2** (schemas) → **Section 3** (list documents) → **Section 4** (classification) → **Section 5** (Quote/MSA parsing) → **Section 5b** (save pr.json, parsed_quotes.json, parsed_msas.json to output/).
3. **Output:** `uploaded`, `classified_documents`, `parsed_quotes`, `parsed_msas`, `pr`; and **output/** contains the parsed JSONs (Section 5b).

**Notebook verification (what's implemented) — through Section 5 (presented today):**  
- **§0:** Environment detection and validation (ENV_MODE=aws or local; required vars checked).  
- **§1:** Paths (input/output/schemas) and LLM (Groq or Bedrock).  
- **§2:** Load schema references (pr, quote, msa; Check 1/2 result schemas).  
- **§3:** List uploaded documents (from input folder or synthetic_data).  
- **§4:** Document categorization — each attachment classified (PR Form, Quotation, Contract, SOW, etc.).  
- **§5:** Quote and MSA parsing (schema-driven, retries, validation, merge). **§5b:** Save pr.json, parsed_quotes.json, parsed_msas.json to output/.

**Not in this notebook:** Structured DB (§6), RAG/vector (§7), orchestrator (§8), compliance checks (§9–12).

---
## 0. Environment detection and validation

**What this does:** Loads `.env` (if present), then sets **ENV_MODE** to `aws` or `local` (from **ENV_MODE** or auto-detect). Validates the **required** environment variables for the chosen mode and raises a clear error if any are missing. Copy **.env.example** to **.env** and fill in the block for your chosen mode. Sets `ENV_MODE`, `USE_BEDROCK`, `USE_GROQ`, `USE_TITAN`, `USE_OPENSEARCH`, `USE_QDRANT`, and `DATABASE_URL` for later cells.

In [1]:
# Load .env first so ENV_MODE and all vars are available
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

import os
import json

def get_env(key: str, default: str = None) -> str:
    v = os.environ.get(key) or os.environ.get(key.upper())
    return (v or "").strip() or default

def detect_env() -> str:
    explicit = get_env("ENV_MODE")
    if explicit and explicit.lower() in ("aws", "local"):
        return explicit.lower()
    use_aws = (get_env("AWS_REGION") or get_env("AWS_PROFILE")) and get_env("BEDROCK_MODEL_ID") and (get_env("OPENSEARCH_ENDPOINT") or get_env("OPENSEARCH_HOST"))
    use_local = get_env("GROQ_API_KEY") and get_env("QDRANT_URL") and get_env("QDRANT_API_KEY")
    if use_aws:
        return "aws"
    if use_local:
        return "local"
    return "local"  # default for notebook

def validate_required_vars(mode: str) -> None:
    if mode == "aws":
        required = {"AWS_REGION or AWS_PROFILE": get_env("AWS_REGION") or get_env("AWS_PROFILE"), "BEDROCK_MODEL_ID": get_env("BEDROCK_MODEL_ID"), "OPENSEARCH_ENDPOINT or OPENSEARCH_HOST": get_env("OPENSEARCH_ENDPOINT") or get_env("OPENSEARCH_HOST")}
    else:
        required = {"GROQ_API_KEY": get_env("GROQ_API_KEY"), "QDRANT_URL": get_env("QDRANT_URL"), "QDRANT_API_KEY": get_env("QDRANT_API_KEY")}
    missing = [k for k, v in required.items() if not v]
    if missing:
        raise ValueError(f"ENV_MODE={mode}: missing required env vars. Set these in .env (see .env.example): " + ", ".join(missing))

ENV_MODE = detect_env()
validate_required_vars(ENV_MODE)
print(f"ENV_MODE: {ENV_MODE} (all required vars present)")

# LLM: Bedrock (AWS) or Groq (local)
USE_BEDROCK = ENV_MODE == "aws"
USE_GROQ = ENV_MODE == "local"
# Embeddings / vector store (used in full pipeline)
USE_TITAN = ENV_MODE == "aws"
USE_LOCAL_EMBEDDINGS = ENV_MODE == "local"
USE_OPENSEARCH = ENV_MODE == "aws"
USE_QDRANT = ENV_MODE == "local"
# DB
DATABASE_URL = get_env("DATABASE_URL") or get_env("LOCAL_DB_PATH") or "sqlite:///pr_validation.db"
print(f"Database: {DATABASE_URL}")

ENV_MODE: local (all required vars present)
Database: sqlite:///pr_validation.db


---
## 1. Imports and config

**What this does:** Loads `.env`, sets **BASE_DIR** / **SCHEMAS_DIR** / **INPUT_DIR** / **OUTPUT_DIR**, creates input/output folders if missing, and creates the **LLM** (Groq or Bedrock) used for document classification.

In [2]:
# Optional: load .env if present
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

import os
from pathlib import Path

# Base paths: run from repo root or from document_processing_rag
CWD = Path(os.getcwd())
if (CWD / "schemas").exists():
    BASE_DIR = CWD
elif (CWD / "document_processing_rag" / "schemas").exists():
    BASE_DIR = CWD / "document_processing_rag"
else:
    BASE_DIR = CWD

SCHEMAS_DIR = BASE_DIR / "schemas"
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Fallback if input is empty: synthetic_data/multi item example/live
if (BASE_DIR.parent / "synthetic_data").exists():
    SYNTHETIC_DIR = BASE_DIR.parent / "synthetic_data" / "multi item example" / "live"
else:
    SYNTHETIC_DIR = BASE_DIR / "synthetic_data" / "multi item example" / "live"

print("BASE_DIR:", BASE_DIR)
print("SCHEMAS_DIR:", SCHEMAS_DIR, "exists:", SCHEMAS_DIR.exists())
print("INPUT_DIR:", INPUT_DIR, "exists:", INPUT_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR, "exists:", OUTPUT_DIR.exists())

BASE_DIR: m:\AI_consulting\2025\Bhavin\JSONs\PRtoPOAgent\document_processing_rag
SCHEMAS_DIR: m:\AI_consulting\2025\Bhavin\JSONs\PRtoPOAgent\document_processing_rag\schemas exists: True
INPUT_DIR: m:\AI_consulting\2025\Bhavin\JSONs\PRtoPOAgent\document_processing_rag\input exists: True
OUTPUT_DIR: m:\AI_consulting\2025\Bhavin\JSONs\PRtoPOAgent\document_processing_rag\output exists: True


In [3]:
# LangChain and LLM
try:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_core.output_parsers import JsonOutputParser
except ImportError:
    raise ImportError("Install: pip install langchain-core")

if USE_GROQ:
    try:
        from langchain_groq import ChatGroq
        llm = ChatGroq(
            model=os.environ.get("LOCAL_LLM_MODEL", "llama-3.1-8b-instant"),
            api_key=os.environ.get("GROQ_API_KEY"),
            temperature=0.1,
        )
        print("LLM: Groq (local)")
    except ImportError:
        raise ImportError("Install: pip install langchain-groq")
elif USE_BEDROCK:
    try:
        from langchain_aws import ChatBedrock
        llm = ChatBedrock(
            model_id=os.environ.get("BEDROCK_MODEL_ID", "anthropic.claude-3-5-haiku-20241022-v2:0"),
            region_name=os.environ.get("AWS_REGION", "us-east-1"),
            temperature=0.1,
        )
        print("LLM: Bedrock (AWS)")
    except ImportError:
        raise ImportError("Install: pip install langchain-aws boto3")
else:
    raise RuntimeError("Set GROQ_API_KEY (local) or AWS + BEDROCK_MODEL_ID (production)")

c:\Users\mauli\anaconda3\envs\pr2po\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM: Groq (local)


---
## 2. Load schema references

**What this does:** Loads schema JSONs from `schemas/`, including the single **PR template** (`pr.json`: header + attachments + line_items). Used for classification prompt structure; full pipeline uses these for parsing and checks later.

In [4]:
def load_schema(name: str) -> dict:
    path = SCHEMAS_DIR / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(str(path))
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_check_result(check_num: int) -> dict:
    if check_num == 1:
        path = SCHEMAS_DIR / "compliance_checks" / "check_01_attachment_existence_classification" / "check_01_result.json"
    elif check_num == 2:
        path = SCHEMAS_DIR / "compliance_checks" / "check_02_document_validity_applicability" / "check_02_result.json"
    else:
        raise ValueError(f"Check {check_num} schema not yet added")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# Load extraction schemas (for parsing in full pipeline). PR is single combined template: header + attachments + line_items.
schemas = {}
for name in ["pr", "quote", "msa"]:
    try:
        schemas[name] = load_schema(name)
        print(f"Loaded schema: {name}")
    except FileNotFoundError as e:
        print(f"Skip {name}: {e}")

# Check result schemas (for full pipeline)
check1_schema = load_check_result(1)
check2_schema = load_check_result(2)
print("Check 1 & 2 result schemas loaded.")

Loaded schema: pr
Loaded schema: quote
Loaded schema: msa
Check 1 & 2 result schemas loaded.


---
## 3. Document upload

**What this does:** Lists documents in the input folder (or synthetic_data) — PDF, Word, Excel. **Output:** `uploaded` (list of path, filename, size_bytes).

In [5]:
def list_uploaded_documents(source_dir: Path = None) -> list[dict]:
    """List documents available for processing. Each item: {path, filename, size_bytes}."""
    source_dir = source_dir or INPUT_DIR
    if not source_dir.exists():
        source_dir = SYNTHETIC_DIR
    if not source_dir.exists():
        return []
    out = []
    for f in source_dir.iterdir():
        if f.is_file() and f.suffix.lower() in (".pdf", ".docx", ".xlsx", ".doc", ".xls"):
            out.append({
                "path": str(f),
                "filename": f.name,
                "size_bytes": f.stat().st_size,
            })
    return sorted(out, key=lambda x: x["filename"])

uploaded = list_uploaded_documents()
for u in uploaded:
    print(u["filename"], "(", u["size_bytes"], "bytes )")
if not uploaded:
    print("No documents found. Place PDF/Word/Excel in the input folder (document_processing_rag/input) or in synthetic_data/.../live.")

01_PR_Header_and_LineItems.xlsx ( 9436 bytes )
02_CDW_Quote_Q25-0847.pdf ( 6118 bytes )
02b_MSA_CDW_2024_0156_Executed.pdf ( 8365 bytes )


---
## 4. Document categorization

**What this does:** Classifies **each uploaded attachment** into one document type: **PR Form** (Excel with PR Header/Line Items/Attachments — one or three sheets), Quotation, Contract, SOW, Service Agreement, Invoice, BidSummary, Justification, Spec, Other. Uses sheet names and column headers so PR form Excel is not miscategorized. **Output:** `classified_documents` — one dict per file with `filename`, `document_type`, `confidence`, `reason`. This is the gatekeeper step; the full pipeline uses this to route Quote vs MSA parsing and for compliance checks.

In [6]:
def extract_text_from_file(filepath: str, max_chars: int = 2000) -> str:
    """Extract first max_chars of text for classification hint. Excel: sheet names + headers + first rows (supports single-sheet or multi-sheet PR Header/Line Items/Attachments)."""
    path = Path(filepath)
    suffix = path.suffix.lower()
    try:
        if suffix == ".pdf":
            try:
                import pypdf
                reader = pypdf.PdfReader(filepath)
                text = ""
                for i, page in enumerate(reader.pages):
                    if i >= 3:
                        break
                    text += page.extract_text() or ""
                return (text or path.name)[:max_chars]
            except ImportError:
                return path.name
        if suffix in (".docx", ".doc"):
            try:
                import docx
                doc = docx.Document(filepath)
                return "\n".join(p.text for p in doc.paragraphs[:30])[:max_chars]
            except ImportError:
                return path.name
        if suffix == ".xlsx":
            try:
                from openpyxl import load_workbook
                wb = load_workbook(filepath, read_only=False, data_only=True)
                parts = [f"Sheets: {', '.join(wb.sheetnames)}"]
                for sheet_name in wb.sheetnames[:5]:
                    ws = wb[sheet_name]
                    rows = list(ws.iter_rows(max_row=4, values_only=True))
                    if rows:
                        header = " | ".join(str(c) if c is not None else "" for c in rows[0][:20])
                        parts.append(f"[{sheet_name}] Headers: {header}")
                        for r in rows[1:4]:
                            parts.append(" | ".join(str(c) if c is not None else "" for c in r[:20]))
                return ("\n".join(parts) or path.name)[:max_chars]
            except Exception:
                return path.name
        if suffix == ".xls":
            return path.name
    except Exception as e:
        return path.name + " " + str(e)[:200]
    return path.name

# Classification system prompt includes JSON example. PR Form = PR data (Excel).
CLASSIFY_EXAMPLE = json.dumps({"document_type": "PR Form", "confidence": 0.95, "reason": "Excel with sheets PR Header, PR Line Items, PR Attachments; columns PR Number, PR Status, Total Estimated Value."})
CLASSIFY_SYSTEM = """You are a procurement document classifier. Given the file name and a short excerpt, classify into exactly one type: PR Form, Quotation, Contract, SOW, Service Agreement, Invoice, BidSummary, Justification, Spec, Other.
PR Form: Excel or workbook with sheet(s) named 'PR Header', 'PR Line Items', 'PR Attachments' OR columns/headers like PR Number, PR Status, Requestor Name, Total Estimated Value, Line Items, Ship-To Address. This is the purchase requisition form data (header, line items, attachments). Can be one sheet with all columns or three separate sheets. NOT a quotation.
Quotation: Quote, Quotation, Valid Until, line items with prices, supplier letterhead.
Contract: Agreement, Contract, Master Service Agreement, parties, effective/expiry.
SOW: Statement of Work, Scope of Work, deliverables, milestones.
Invoice: Invoice, Bill To, Payment Due.
Respond with JSON only. Example output: """ + CLASSIFY_EXAMPLE

def classify_document(llm, filepath: str, filename: str, excerpt: str = None) -> dict:
    excerpt = excerpt or extract_text_from_file(filepath)
    prompt = f"File: {filename}\n\nExcerpt:\n{excerpt[:1500]}\n\nClassify. JSON only."
    msg = [SystemMessage(content=CLASSIFY_SYSTEM), HumanMessage(content=prompt)]
    out = llm.invoke(msg)
    text = out.content if hasattr(out, "content") else str(out)
    try:
        if "```" in text:
            text = text.split("```")[1].replace("json", "").strip()
        return json.loads(text)
    except json.JSONDecodeError:
        return {"document_type": "Other", "confidence": 0.5, "reason": "Parse failed"}

# Run classification on uploaded docs
classified_documents = []
for u in uploaded:
    result = classify_document(llm, u["path"], u["filename"])
    result["filename"] = u["filename"]
    classified_documents.append(result)
    print(u["filename"], "->", result.get("document_type"), "(", result.get("confidence"), ")")

01_PR_Header_and_LineItems.xlsx -> PR Form ( 0.99 )
02_CDW_Quote_Q25-0847.pdf -> Quotation ( 0.95 )
02b_MSA_CDW_2024_0156_Executed.pdf -> Contract ( 0.99 )


**What the parsing implementation does:**

1. **System prompt** — The LLM has a fixed role and rules; prompts embed the full schema **example** so output matches the template.
2. **Schema-driven extraction** — The LLM is told exactly which fields to extract (from the schema); this reduces omitted fields and invalid JSON.
3. **Validation** — Parsed output is checked against the schema; if required fields are missing or invalid, we retry with the list of errors.
4. **Retry and merge** — Up to 3 attempts; each retry sends the validation errors to the LLM and **merge_deep** combines the new extraction with the previous one.

**In this notebook:** **validate_parsed_output** and **merge_deep** are used inside **run_extraction_with_retries**.

**Helpers used by Section 5 (run the cell below first).**
- **get_schema_field_list(schema)** — Collects property names from the schema for prompt hints.
- **_schema_for_validation(schema)** — Strips non–JSON Schema keys (e.g. `example`) so `jsonschema.validate()` can run.
- **validate_parsed_output(data, schema)** — Checks required fields and optional `jsonschema`; returns `(valid: bool, errors: list)`. Used inside the retry loop.
- **merge_deep(base, update)** — Deep-merges `update` into `base` (fills missing or empty values). Used to combine the initial parse with the retry parse.

In [7]:
# Schema-driven extraction: field list + validation (production tool)
def get_schema_field_list(schema: dict, prefix: str = "") -> list[str]:
    """Recursively collect property names from a JSON schema for prompt guidance."""
    props = schema.get("properties", {})
    out = []
    for key, val in props.items():
        if isinstance(val, dict) and "properties" in val and key not in ("bill_to", "ship_to", "line_items"):
            out.extend(get_schema_field_list(val, prefix + key + "."))
        elif isinstance(val, dict) and "items" in val and "properties" in val.get("items", {}):
            out.append(f"{prefix}{key}[] (each: {list(val['items'].get('properties', {}).keys())})")
        else:
            out.append(prefix + key)
    return out

def _schema_for_validation(schema: dict) -> dict:
    """Return a copy of schema without non-jsonschema keys (e.g. example) for validate()."""
    s = {k: v for k, v in schema.items() if k not in ("example", "title", "description")}
    if "properties" in s:
        s["properties"] = {k: v for k, v in s["properties"].items()}
    return s

def validate_parsed_output(data: dict, schema: dict) -> tuple[bool, list[str]]:
    """Returns (valid, list of missing/invalid field messages). Production tool for extract loop."""
    errors = []
    required = schema.get("required", [])
    for r in required:
        if r not in data or data[r] is None or data[r] == "":
            errors.append(f"Missing required: {r}")
    if "line_items" in schema.get("properties", {}) and isinstance(data.get("line_items"), list):
        item_schema = schema.get("properties", {}).get("line_items", {}).get("items", {})
        for i, item in enumerate(data["line_items"]):
            for req in item_schema.get("required", []):
                if req not in item or item[req] is None:
                    errors.append(f"line_items[{i}] missing: {req}")
    try:
        import jsonschema
        jsonschema.validate(instance=data, schema=_schema_for_validation(schema))
    except ImportError:
        pass
    except Exception as e:
        if "ValidationError" in type(e).__name__:
            errors.append(str(e))
        elif not errors:
            errors.append(str(e))
    return (len(errors) == 0, errors)

def merge_deep(base: dict, update: dict) -> dict:
    """Deep merge update into base (update fills missing or empty in base). For retry merge."""
    out = dict(base)
    for k, v in update.items():
        if v is None or v == "":
            continue
        if k not in out or out[k] is None or out[k] == "":
            out[k] = v
        elif isinstance(out[k], dict) and isinstance(v, dict):
            out[k] = merge_deep(out[k], v)
        elif isinstance(out[k], list) and isinstance(v, list) and v:
            if not out[k]:
                out[k] = v
            else:
                for i, item in enumerate(v):
                    if i < len(out[k]) and isinstance(out[k][i], dict) and isinstance(item, dict):
                        out[k][i] = merge_deep(out[k][i], item)
                    elif i >= len(out[k]):
                        out[k].append(item)
    return out

---
## 5. Document parsing (JSON templates + retries)

**What this does:** For each attachment classified as **Quotation** or **Contract/MSA**, runs **production extraction** with retries, validation, and merge.

- **System prompts:** `QUOTE_EXTRACT_SYSTEM` and `MSA_EXTRACT_SYSTEM` are built from `schemas/quote.json` and `schemas/msa.json` and **embed the full `example`** so the LLM outputs JSON that matches the template.
- **`_parse_json_from_llm(text_out)`** — Strips markdown code fences and parses JSON from the LLM response; returns `None` on failure.
- **`run_extraction_with_retries(llm, doc_type, filepath, filename, max_retries=3)`** — Loop: build prompt, call LLM, parse JSON, run **`validate_parsed_output(parsed, schema)`**. If invalid and retries left, call LLM again with the error list and **`merge_deep(parsed, retry_parsed)`**; repeat until valid or max retries. Returns the merged parsed dict or a minimal fallback.
- **`parse_quote_with_llm`** / **`parse_msa_with_llm`** — Thin wrappers that call `run_extraction_with_retries` with `doc_type` `"quote"` or `"msa"`.

**Output:** `parsed_quotes` and `parsed_msas` — lists of extracted JSONs, one per classified quote or MSA attachment.

In [8]:
# System prompts: schemas/quote.json and schemas/msa.json (loaded in §2); example embeds in prompt, required drives validation
_quote_schema = schemas.get("quote", {})
_quote_example = json.dumps(_quote_schema.get("example", {}), indent=2)[:3800]
QUOTE_EXTRACT_SYSTEM = """You are a quote parser for procurement. Extract structured data from supplier quotation text into a JSON object. Your output MUST match this structure and types. Extract every field that appears in the document.

Example output (adapt values to the document; keep structure and types):
""" + _quote_example + """

Rules: Output valid JSON only. No markdown. Dates YYYY-MM-DD. Numbers as numbers. Required: quote_number, quote_date, currency, line_items (each with line_number, quantity, unit_of_measure, unit_price, extended_price)."""

_msa_schema = schemas.get("msa", {})
_msa_example = json.dumps(_msa_schema.get("example", {}), indent=2)[:3800]
MSA_EXTRACT_SYSTEM = """You are an MSA/contract parser for procurement. Extract structured data from Master Service Agreement or contract text. Your output MUST match this structure and types.

Example output (adapt values to the document; keep structure and types):
""" + _msa_example + """

Rules: Output valid JSON only. No markdown. Dates YYYY-MM-DD. Required: agreement_id, effective_date, buyer_name, supplier_name."""

def _parse_json_from_llm(text_out: str) -> dict | None:
    if "```" in text_out:
        text_out = text_out.split("```")[1].replace("json", "").strip()
    try:
        return json.loads(text_out)
    except json.JSONDecodeError:
        return None

def run_extraction_with_retries(llm, doc_type: str, filepath: str, filename: str, max_retries: int = 3) -> dict:
    """Production: retry loop + validation tool + merge. doc_type in ('quote', 'msa')."""
    schema = _quote_schema if doc_type == "quote" else _msa_schema
    system_prompt = QUOTE_EXTRACT_SYSTEM if doc_type == "quote" else MSA_EXTRACT_SYSTEM
    max_chars = 6000 if doc_type == "quote" else 5000
    text = extract_text_from_file(filepath, max_chars=max_chars)
    fallback = (
        {"source_file_name": filename, "quote_number": "unknown", "quote_date": "", "currency": "USD", "line_items": []}
        if doc_type == "quote"
        else {"source_file_name": filename, "agreement_id": "unknown", "effective_date": "", "buyer_name": "", "supplier_name": ""}
    )
    example_snippet = json.dumps(schema.get("example", {}), indent=2)[:2800]
    parsed = None
    errors = []
    for attempt in range(max_retries):
        if attempt == 0:
            prompt = f"Extract data into JSON. Example structure:\n{example_snippet}\n\nDocument (filename: {filename}):\n{text}"
        else:
            prompt = f"Document (excerpt):\n{text[:2500]}\n\nValidation failed (attempt {attempt + 1}/{max_retries}): {errors}\nReturn complete valid JSON that fixes these. Output only JSON."
        msg = [SystemMessage(content=system_prompt), HumanMessage(content=prompt)]
        out = llm.invoke(msg)
        text_out = out.content if hasattr(out, "content") else str(out)
        parsed = _parse_json_from_llm(text_out)
        if not parsed:
            continue
        parsed.setdefault("source_file_name", filename)
        valid, errors = validate_parsed_output(parsed, schema)
        if valid:
            break
        if attempt < max_retries - 1:
            retry_prompt = f"Document (excerpt):\n{text[:2500]}\n\nValidation failed (attempt {attempt + 1}/{max_retries}): {errors}\nReturn complete valid JSON that fixes these. Output only JSON."
            retry_out = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=retry_prompt)])
            retry_parsed = _parse_json_from_llm(retry_out.content if hasattr(retry_out, "content") else str(retry_out))
            if retry_parsed:
                parsed = merge_deep(parsed, retry_parsed)
                valid, errors = validate_parsed_output(parsed, schema)
                if valid:
                    break
    return parsed if parsed else fallback

def parse_quote_with_llm(llm, filepath: str, filename: str) -> dict:
    return run_extraction_with_retries(llm, "quote", filepath, filename, max_retries=3)

def parse_msa_with_llm(llm, filepath: str, filename: str) -> dict:
    return run_extraction_with_retries(llm, "msa", filepath, filename, max_retries=3)

# Parse quotes and MSAs from classified docs (production: retries + validation + merge)
parsed_quotes = []
parsed_msas = []
for u, cl in zip(uploaded, classified_documents):
    dt = cl.get("document_type", "Other")
    if dt == "Quotation":
        parsed_quotes.append(parse_quote_with_llm(llm, u["path"], u["filename"]))
    elif dt in ("Contract", "Master Service Agreement") or "Contract" in dt:
        parsed_msas.append(parse_msa_with_llm(llm, u["path"], u["filename"]))

print("Parsed quotes:", len(parsed_quotes))
print("Parsed MSAs:", len(parsed_msas))

Parsed quotes: 1
Parsed MSAs: 1


In [9]:
# Stub PR from single template (schemas/pr.json: header + attachments + line_items).
pr = schemas.get("pr", {}).get("example", {})
if not pr:
    pr = {"header": {}, "attachments": {"attachment_count": 0, "attachments": []}, "line_items": []}
pr_header = pr.get("header", {})
pr_line_items = {"pr_number": pr_header.get("pr_number", "PR-2026-007842"), "line_items": pr.get("line_items", [])}
DOC_TYPE_TO_ATTACHMENT_CLASS = {
    "PR Form": "Other",
    "Quotation": "Supplier Quotation",
    "Contract": "Master Service Agreement / Contract Services",
    "Master Service Agreement": "Master Service Agreement / Contract Services",
    "Service Agreement": "Master Service Agreement / Contract Services",
    "SOW": "SOW",
    "Invoice": "Invoice",
    "BidSummary": "Competitive Quote Comparison",
    "Justification": "Technical Evaluation / Business Case",
    "Spec": "Other",
}
pr_attachments = {
    "attachment_count": len(uploaded),
    "attachments": [
        {
            "attachment_number": i + 1,
            "file_name": u["filename"],
            "file_type": "PDF" if u["filename"].endswith(".pdf") else "Excel" if u["filename"].lower().endswith((".xlsx", ".xls")) else "Word" if u["filename"].lower().endswith((".docx", ".doc")) else "Other",
            "document_classification": DOC_TYPE_TO_ATTACHMENT_CLASS.get(cl.get("document_type"), "Other"),
            "classification_confidence": cl.get("confidence", 0.8),
        }
        for i, (u, cl) in enumerate(zip(uploaded, classified_documents))
    ],
}
pr = {"header": pr_header, "attachments": pr_attachments, "line_items": pr_line_items["line_items"]}
print("PR header pr_number:", pr_header.get("pr_number"))
print("PR line items count:", len(pr_line_items.get("line_items", [])))
print("PR attachments count:", pr_attachments["attachment_count"])

PR header pr_number: PR-2026-007842
PR line items count: 2
PR attachments count: 3


---
## 5b. Save parsed outputs to output folder

**What this does:** Writes parsed data to the **output** folder immediately after parsing (and after building the combined PR). This way **pr.json**, **parsed_quotes.json**, and **parsed_msas.json** are available before database persistence, RAG ingest, or compliance checks.

**Output:** Files under `document_processing_rag/output/`: **pr.json**, **parsed_quotes.json**, **parsed_msas.json**.

In [10]:
# Save parsed outputs right after parsing (before DB, RAG, or checks)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_DIR / "pr.json", "w", encoding="utf-8") as f:
    json.dump(pr, f, indent=2)
with open(OUTPUT_DIR / "parsed_quotes.json", "w", encoding="utf-8") as f:
    json.dump(parsed_quotes, f, indent=2)
with open(OUTPUT_DIR / "parsed_msas.json", "w", encoding="utf-8") as f:
    json.dump(parsed_msas, f, indent=2)
print("Parsed outputs saved to", OUTPUT_DIR)
print("  - pr.json, parsed_quotes.json, parsed_msas.json")

Parsed outputs saved to m:\AI_consulting\2025\Bhavin\JSONs\PRtoPOAgent\document_processing_rag\output
  - pr.json, parsed_quotes.json, parsed_msas.json
